# LongEval 2026 Task 4 RAG Baseline — BM25 + English Embeddings

This notebook is a **simple baseline** for the official 2026 Task 4 setup:

- use the official Task 4 queries
- use only the 10 candidate document ids per query
- rank candidate document chunks with **BM25 + English semantic embeddings**
- generate **one answer per query**
- attach citation indices into the provided `references` list


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip -q install ir-datasets-longeval pyarrow pandas numpy rank-bm25 sentence-transformers transformers accelerate bitsandbytes nltk scikit-learn tqdm pyyaml

In [ ]:
import os
import gc
import re
import json
import math
import textwrap
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple, Any
import unicodedata

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import sent_tokenize

nltk.download("punkt", quiet=True)

warnings.filterwarnings("ignore")

In [ ]:
import nltk

for pkg in ["punkt", "punkt_tab"]:
    try:
        nltk.data.find(f"tokenizers/{pkg}")
    except LookupError:
        nltk.download(pkg)

In [ ]:
# =========================
# User config
# =========================

TEAM_ID = "LongEval DS@GT"
RUN_ID = "baseline-bm25-bge-gemma4-31b"
# RUN_ID = "baseline-bm25-bge-Qwen2-7b"
RUN_TYPE = "automatic"

# Preferred official dataset tag
DATASET_TAG = "longeval-sci-2026/clef-2026/rag"

# Local files (optional fallbacks)
LOCAL_QUERY_JSONL = "/content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Data/Test Data/task4_longeval_rag-query_docids.jsonl"
LOCAL_DOC_PARQUET = "/content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Data/task4_candidate_documents.parquet"
USE_LOCAL_PARQUET_DOCS = True

# Output locations
OUTPUT_DIR = Path("/content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_JSONL = OUTPUT_DIR / f"{RUN_ID}.jsonl"
SYSTEM_YAML = OUTPUT_DIR / f"{RUN_ID}.yaml"

# =========================
# Retrieval / evidence controls
# =========================

# Because there are only 10 candidate docs, we can inspect many chunks per doc.
PASSAGE_WINDOW_SENTENCES = 4
PASSAGE_STRIDE_SENTENCES = 2
MAX_PASSAGES_PER_DOC = 120

DO_SAMPLE = False

# Evidence passed to the LLM
TOP_PASSAGES_FINAL = 20

# Citation strategy:
# There is no required minimum citation count.
# Cite only documents whose chunks meaningfully support the generated answer.
# In practice, most scientific answers should usually have 1-2 citations,
# but this is evidence-driven rather than hardcoded.
CLAIM_SUPPORT_THRESHOLD = 0.28
CLAIM_SECOND_SUPPORT_THRESHOLD = 0.30
CLAIM_SECOND_RELATIVE_THRESHOLD = 0.60
CLAIM_TOP_K = 3

# Keep answer concise to improve ROUGE/BERTScore and reduce unsupported claims.
MAX_SENTENCES_IN_ANSWER = 3
MAX_ANSWER_CHARS = 450
MAX_CHARS_PER_EVIDENCE = 350
MAX_INPUT_TOKENS = 4096

# =========================
# Models
# =========================

# Queries are English, so use an English embedding model rather than a multilingual one.
USE_EMBED_RERANK = True
EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"   # use bge-small if Colab RAM is tight

# The baseline comparison is BM25 + semantic embedding ranking
USE_LLM = True
GEN_MODEL_NAME = "google/gemma-4-31B-it"
# GEN_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Generation settings
MAX_NEW_TOKENS = 150
DO_SAMPLE = False


## Load the official queries

Preferred path: load from `ir-datasets-longeval`.  
Fallback: read the provided `task4_longeval_rag-query_docids.jsonl`.

In [ ]:
def normalize_text(x: Any) -> str:
    x = "" if x is None else str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_unicode_answer(text: str) -> str:
    return unicodedata.normalize("NFKC", normalize_text(text))

def contains_foreign_script(text: str) -> bool:
    """Catch obvious non-English script leakage while allowing normal scientific symbols."""
    return bool(re.search(r"[\u0400-\u04FF\u4E00-\u9FFF\u3040-\u30FF\uAC00-\uD7AF]", text))

def try_import_ir_dataset():
    try:
        from ir_datasets_longeval import load
        return load
    except Exception as e:
        print("Could not import ir_datasets_longeval:", e)
        return None

def load_task4_queries(dataset_tag: str, local_query_jsonl: Optional[str] = None) -> pd.DataFrame:
    # 1) Try official ir-datasets
    load_fn = try_import_ir_dataset()
    if load_fn is not None:
        try:
            ds = load_fn(dataset_tag)
            rows = []
            for q in ds.queries_iter():
                qid = str(getattr(q, "query_id", getattr(q, "qid", "")))
                qtext = getattr(q, "query", None)
                if qtext is None:
                    # different objects in different wrappers
                    qtext = getattr(q, "text", None)
                if qtext is None and hasattr(q, "default_text"):
                    qtext = q.default_text()
                doc_ids = getattr(q, "doc_ids", None)
                if doc_ids is None and hasattr(q, "_asdict"):
                    doc_ids = q._asdict().get("doc_ids")
                if doc_ids is None:
                    raise ValueError("Could not find doc_ids on query object.")
                rows.append({
                    "query_id": str(qid),
                    "query": normalize_text(qtext),
                    "doc_ids": [str(x) for x in doc_ids],
                })
            qdf = pd.DataFrame(rows)
            if len(qdf) > 0:
                print(f"Loaded {len(qdf)} queries from ir-datasets-longeval.")
                return qdf
        except Exception as e:
            print("Official dataset query load failed, falling back to local JSONL.", e)

    # 2) Fallback local jsonl
    if local_query_jsonl and Path(local_query_jsonl).exists():
        rows = []
        with open(local_query_jsonl, "r") as f:
            for line in f:
                obj = json.loads(line)
                rows.append({
                    "query_id": str(obj["query_id"]),
                    "query": normalize_text(obj.get("query") or obj.get("question")),
                    "doc_ids": [str(x) for x in obj["doc_ids"]],
                })
        qdf = pd.DataFrame(rows)
        print(f"Loaded {len(qdf)} queries from local JSONL.")
        return qdf

    raise FileNotFoundError("Could not load Task 4 queries from ir-datasets or local JSONL.")

queries_df = load_task4_queries(DATASET_TAG, LOCAL_QUERY_JSONL)
queries_df.head()

Loaded 47 queries from ir-datasets-longeval.


,query_id,query,doc_ids
0,aa42e210a361571ff4d1fad892b75d15,How can a device avoid futile access attempts ...,"[275699672, 275699915, 122371639, 173704007, 1..."
1,46395a3cf66a9f6a75c89354410d1493,How should pesticide class–specific findings s...,"[290464296, 293060390, 283174876, 276607117, 3..."
2,5f07a7f0a83de807eee3213560a3467c,What tradeoff arises when applying exact gradi...,"[156685367, 303260254, 291415237, 293226721, 4..."
3,804711f54939af9622c3c7614da85298,How well do trial-specific early composite ben...,"[309224898, 299770278, 308409429, 292908773, 3..."
4,c3d5d202f8ab3d130a90684fd7308f64,Does healing fingertip ischemic lesions affect...,"[293025457, 296429828, 305621459, 11741532, 31..."


## Load candidate documents

This notebook supports two sources:

1. **Official docs store** from `ir-datasets-longeval`
2. **Local parquet** fallback

The official collection is incremental across snapshots, and the 2026 extension notes that the RAG questions are tied to the most recent snapshot (`snapshot-3`).


In [ ]:
# ============================================================
# Build a tiny parquet containing only Task 4 candidate docs
# ============================================================

import os
import json
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pyarrow.parquet as pq
from pathlib import Path

FULL_LOCAL_DOC_PARQUET = Path("/content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Data/longeval_documents.parquet")
CANDIDATE_DOC_PARQUET = Path("/content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Data/task4_candidate_documents.parquet")

all_candidate_doc_ids = sorted({
    str(doc_id)
    for _, qrow in queries_df.iterrows()
    for doc_id in qrow["doc_ids"]
})

print("Unique candidate doc ids needed:", len(all_candidate_doc_ids))
print("Full parquet exists:", FULL_LOCAL_DOC_PARQUET.exists())
print("Full parquet size GB:", round(os.path.getsize(FULL_LOCAL_DOC_PARQUET) / (1024**3), 3))

if not CANDIDATE_DOC_PARQUET.exists():
    dataset = ds.dataset(str(FULL_LOCAL_DOC_PARQUET), format="parquet")

    print("Columns:", dataset.schema.names)

    # Only load the columns needed for Task 4.
    keep_cols = [c for c in ["doc_id", "original_text", "metadata"] if c in dataset.schema.names]
    print("Keeping columns:", keep_cols)

    # Filter at the Arrow level, before converting to pandas.
    filter_expr = pc.field("doc_id").isin(all_candidate_doc_ids)

    table = dataset.to_table(
        columns=keep_cols,
        filter=filter_expr
    )

    print("Rows retrieved:", table.num_rows)

    pq.write_table(table, CANDIDATE_DOC_PARQUET)
    print("Saved small candidate parquet:", CANDIDATE_DOC_PARQUET)
    print("Small parquet size MB:", round(os.path.getsize(CANDIDATE_DOC_PARQUET) / (1024**2), 2))
else:
    print("Small candidate parquet already exists:", CANDIDATE_DOC_PARQUET)
    print("Small parquet size MB:", round(os.path.getsize(CANDIDATE_DOC_PARQUET) / (1024**2), 2))

Unique candidate doc ids needed: 468
Full parquet exists: True
Full parquet size GB: 6.873
Small candidate parquet already exists: /content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Data/task4_candidate_documents.parquet
Small parquet size MB: 5.36


In [ ]:
@dataclass
class DocRecord:
    doc_id: str
    title: str = ""
    abstract: str = ""
    fulltext: str = ""
    published_date: str = ""
    updated_date: str = ""
    raw: Optional[dict] = None

    @property
    def combined_text(self) -> str:
        parts = [self.title, self.abstract, self.fulltext]
        parts = [normalize_text(p) for p in parts if normalize_text(p)]
        return "\n".join(parts).strip()

class OfficialDocsAccessor:
    def __init__(self, dataset_tag: str):
        load_fn = try_import_ir_dataset()
        if load_fn is None:
            raise ImportError("ir_datasets_longeval is not available.")
        self.dataset = load_fn(dataset_tag)
        self.store = self.dataset.docs_store()

    def get(self, doc_id: str) -> Optional[DocRecord]:
        try:
            d = self.store.get(str(doc_id))
            if d is None:
                return None
            # robust extraction because different wrappers use different object types
            if hasattr(d, "_asdict"):
                x = d._asdict()
            elif isinstance(d, dict):
                x = d
            else:
                x = {k: getattr(d, k) for k in dir(d) if not k.startswith("_") and not callable(getattr(d, k))}
            return DocRecord(
                doc_id=str(doc_id),
                title=normalize_text(x.get("title", "")),
                abstract=normalize_text(x.get("abstract", "")),
                fulltext=normalize_text(x.get("fulltext", x.get("body", x.get("text", "")))),
                published_date=str(x.get("publishedDate", x.get("published_date", "")) or ""),
                updated_date=str(x.get("updatedDate", x.get("updated_date", "")) or ""),
                raw=x,
            )
        except Exception:
            return None

class ParquetDocsAccessor:
    def __init__(self, parquet_path: str):
        self.parquet_path = parquet_path
        self.df = pd.read_parquet(parquet_path)

        cols = {c.lower(): c for c in self.df.columns}

        self.id_col = cols.get("doc_id") or cols.get("id")
        self.title_col = cols.get("title")
        self.abstract_col = cols.get("abstract")
        self.fulltext_col = (
            cols.get("original_text")
            or cols.get("fulltext")
            or cols.get("full_text")
            or cols.get("text")
            or cols.get("contents")
            or cols.get("content")
            or cols.get("body")
            or cols.get("body_text")
            or cols.get("paper_text")
            or cols.get("pdf_text")
            or cols.get("plain_text")
        )
        self.metadata_col = cols.get("metadata")

        if self.id_col is None:
            raise ValueError(f"Could not find a doc id column in {self.df.columns.tolist()}")

        if self.fulltext_col is None:
            raise ValueError(
                f"Could not find a full-text column. Columns are: {self.df.columns.tolist()}"
            )

        self.df[self.id_col] = self.df[self.id_col].astype(str)
        self.lookup = self.df.set_index(self.id_col, drop=False)

        print("Parquet rows:", len(self.df))
        print("ID column:", self.id_col)
        print("Full-text column:", self.fulltext_col)
        print("Metadata column:", self.metadata_col)

    def _parse_metadata(self, meta):
        if meta is None or (isinstance(meta, float) and pd.isna(meta)):
            return {}

        if isinstance(meta, dict):
            return meta

        if isinstance(meta, str):
            try:
                return json.loads(meta)
            except Exception:
                return {"raw_metadata": meta}

        return {"raw_metadata": str(meta)}

    def get(self, doc_id: str) -> Optional[DocRecord]:
        doc_id = str(doc_id)

        if doc_id not in self.lookup.index:
            return None

        row = self.lookup.loc[doc_id]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]

        meta = self._parse_metadata(row[self.metadata_col]) if self.metadata_col else {}

        title = ""
        abstract = ""

        if self.title_col:
            title = normalize_text(row[self.title_col])
        if not title:
            title = normalize_text(meta.get("title", ""))

        if self.abstract_col:
            abstract = normalize_text(row[self.abstract_col])
        if not abstract:
            abstract = normalize_text(meta.get("abstract", ""))

        fulltext = normalize_text(row[self.fulltext_col]) if self.fulltext_col else ""

        published_date = normalize_text(
            meta.get("publishedDate")
            or meta.get("published_date")
            or meta.get("createdDate")
            or ""
        )
        updated_date = normalize_text(meta.get("updatedDate") or meta.get("updated_date") or "")

        raw = {
          "doc_id": doc_id,
          "metadata": meta,
        }

        return DocRecord(
            doc_id=doc_id,
            title=title,
            abstract=abstract,
            fulltext=fulltext,
            published_date=published_date,
            updated_date=updated_date,
            raw=raw,
        )

def build_docs_accessor(dataset_tag: str, local_doc_parquet: Optional[str] = None):
    if USE_LOCAL_PARQUET_DOCS and local_doc_parquet and Path(local_doc_parquet).exists():
        print("Using local parquet document store:", local_doc_parquet)
        return ParquetDocsAccessor(local_doc_parquet)

    try:
        accessor = OfficialDocsAccessor(dataset_tag)
        print("Using official docs_store from ir-datasets-longeval.")
        return accessor
    except Exception as e:
        print("Official docs_store load failed:", e)

    if local_doc_parquet and Path(local_doc_parquet).exists():
        print("Using local parquet fallback:", local_doc_parquet)
        return ParquetDocsAccessor(local_doc_parquet)

    raise FileNotFoundError("No usable document source found.")

docs_accessor = build_docs_accessor(DATASET_TAG, LOCAL_DOC_PARQUET)

Using local parquet document store: /content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Data/task4_candidate_documents.parquet
Parquet rows: 275
ID column: doc_id
Full-text column: original_text
Metadata column: metadata


In [ ]:
# Quick sanity check on the first query's candidate docs
sample = queries_df.iloc[0]
sample_docs = [docs_accessor.get(x) for x in sample.doc_ids]
[(d.doc_id, d.title[:80], len(d.abstract), len(d.fulltext), len(d.combined_text)) for d in sample_docs if d is not None][:10]

[('275699672', '', 0, 1679, 1679),
 ('275699915', '', 0, 1856, 1856),
 ('122371639', '', 0, 13330, 13330),
 ('173704007', '', 0, 1629, 1629),
 ('157915138', '', 0, 1279, 1279),
 ('163088229', '', 0, 36393, 36393),
 ('148414038', '', 0, 40302, 40302),
 ('268301', '', 0, 56254, 56254),
 ('129069349', '', 0, 15127, 15127)]

## Evidence segmentation and ranking

This is the actual baseline retrieval component.

The task already gives us a 10-document candidate set, so we do not perform global corpus retrieval. Instead, we:

1. split each candidate document into title/abstract/body chunks,
2. score each chunk with BM25,
3. score each chunk with an English embedding model,
4. combine the scores,
5. pass the top evidence chunks to the answer generator.


In [ ]:
def simple_tokenize(text: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", normalize_text(text).lower())

def norm01(x):
    arr = np.asarray(x, dtype=float)
    if len(arr) == 0:
        return arr
    lo, hi = float(np.min(arr)), float(np.max(arr))
    if hi <= lo:
        return np.zeros_like(arr, dtype=float)
    return (arr - lo) / (hi - lo)

def chunk_document(doc: DocRecord,
                   window_sentences: int = PASSAGE_WINDOW_SENTENCES,
                   stride_sentences: int = PASSAGE_STRIDE_SENTENCES,
                   max_chunks: int = MAX_PASSAGES_PER_DOC) -> List[dict]:
    chunks = []

    title = normalize_text(doc.title)
    abstract = normalize_text(doc.abstract)
    fulltext = normalize_text(doc.fulltext)

    if title or abstract:
        ta = "\n".join([
            f"Title: {title}" if title else "",
            f"Abstract: {abstract}" if abstract else "",
        ]).strip()
        if ta:
            chunks.append({"kind": "title_abstract", "text": ta})

    base = fulltext
    if not base:
        return chunks[:max_chunks]

    try:
        sents = [s.strip() for s in sent_tokenize(base) if len(s.strip()) >= 35]
    except Exception:
        sents = re.split(r"(?<=[.!?])\s+", base)
        sents = [s.strip() for s in sents if len(s.strip()) >= 35]

    for start in range(0, len(sents), stride_sentences):
        window = " ".join(sents[start:start + window_sentences]).strip()
        if len(window) >= 100:
            chunks.append({"kind": "fulltext_window", "text": window})
        if len(chunks) >= max_chunks:
            break

    out, seen = [], set()
    for ch in chunks:
        key = ch["text"][:250].lower()
        if key not in seen:
            out.append(ch)
            seen.add(key)

    return out[:max_chunks]

def lexical_overlap(query: str, text: str) -> float:
    q = simple_tokenize(query)
    t = simple_tokenize(text)
    if not q or not t:
        return 0.0
    qs, ts = set(q), set(t)
    return len(qs & ts) / max(1, len(qs))

_embedder = None
def get_embedder():
    global _embedder
    if _embedder is None:
        from sentence_transformers import SentenceTransformer
        _embedder = SentenceTransformer(EMBED_MODEL_NAME)
    return _embedder

def rank_candidate_passages(query: str,
                            doc_ids: List[str],
                            accessor,
                            top_k: int = TOP_PASSAGES_FINAL) -> List[dict]:
    rows = []

    for ref_idx, doc_id in enumerate(doc_ids):
        doc = accessor.get(str(doc_id))
        if doc is None:
            continue

        chunks = chunk_document(doc)
        for c_idx, ch in enumerate(chunks):
            text = ch["text"]
            text_for_rank = f"{doc.title}. {text}" if doc.title else text
            rows.append({
                "doc_id": str(doc_id),
                "ref_idx": int(ref_idx),
                "chunk_id": int(c_idx),
                "kind": ch.get("kind", "body"),
                "title": doc.title,
                "text": text,
                "text_for_rank": text_for_rank,
                "lexical": lexical_overlap(query, text_for_rank),
            })

    if not rows:
        return []

    df = pd.DataFrame(rows)

    # BM25 over all candidate chunks.
    tokenized = [simple_tokenize(x) for x in df["text_for_rank"].tolist()]
    bm25 = BM25Okapi(tokenized)
    df["bm25"] = bm25.get_scores(simple_tokenize(query))

    df["lexical_n"] = norm01(df["lexical"].to_numpy(dtype=float))
    df["bm25_n"] = norm01(df["bm25"].to_numpy(dtype=float))

    # Dense semantic similarity.
    if USE_EMBED_RERANK and len(df) > 0:
        try:
            embedder = get_embedder()
            q_emb = embedder.encode([query], normalize_embeddings=True, show_progress_bar=False)
            p_emb = embedder.encode(
                df["text_for_rank"].tolist(),
                normalize_embeddings=True,
                batch_size=32,
                show_progress_bar=False,
            )
            df["embed"] = p_emb @ q_emb[0]
            df["embed_n"] = norm01(df["embed"].to_numpy(dtype=float))
            df["score"] = 0.15 * df["lexical_n"] + 0.45 * df["bm25_n"] + 0.40 * df["embed_n"]
        except Exception as e:
            print("Embedding rerank skipped:", e)
            df["score"] = 0.25 * df["lexical_n"] + 0.75 * df["bm25_n"]
    else:
        df["score"] = 0.25 * df["lexical_n"] + 0.75 * df["bm25_n"]

    # Add a document-level score for later minimum-citation backfill.
    doc_scores = (
        df.groupby("ref_idx")["score"]
          .max()
          .reset_index()
          .rename(columns={"score": "doc_score"})
          .sort_values("doc_score", ascending=False)
    )
    doc_rank = {int(r.ref_idx): i for i, r in enumerate(doc_scores.itertuples(index=False))}
    doc_score = {int(r.ref_idx): float(r.doc_score) for r in doc_scores.itertuples(index=False)}
    df["doc_rank"] = df["ref_idx"].map(doc_rank).fillna(999).astype(int)
    df["doc_score"] = df["ref_idx"].map(doc_score).fillna(0.0)

    df = df.sort_values(["score", "bm25"], ascending=False).reset_index(drop=True)

    # Avoid sending the LLM 18-24 near-duplicate chunks from one document.
    # This is evidence diversification, not citation targeting.
    selected = []
    doc_counts = {}
    MAX_CHUNKS_PER_DOC_FOR_GENERATION = 4

    for _, row in df.iterrows():
        ridx = int(row["ref_idx"])
        if doc_counts.get(ridx, 0) >= MAX_CHUNKS_PER_DOC_FOR_GENERATION:
            continue

        selected.append(row.to_dict())
        doc_counts[ridx] = doc_counts.get(ridx, 0) + 1

        if len(selected) >= top_k:
            break

    # Fill if the cap made the selected set too small.
    if len(selected) < top_k:
        seen = {(int(x["ref_idx"]), int(x["chunk_id"])) for x in selected}
        for _, row in df.iterrows():
            key = (int(row["ref_idx"]), int(row["chunk_id"]))
            if key not in seen:
                selected.append(row.to_dict())
            if len(selected) >= top_k:
                break

    return selected


In [ ]:
# Inspect evidence for one query
example_ranked = rank_candidate_passages(
    query=queries_df.iloc[0]["query"],
    doc_ids=queries_df.iloc[0]["doc_ids"],
    accessor=docs_accessor,
    top_k=5,
)
example_ranked[:2]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'doc_id': '122371639',
  'ref_idx': 2,
  'chunk_id': 5,
  'kind': 'fulltext_window',
  'title': '',
  'text': 'As such, if the UE encounters incorrect or non-optimized configurations for a cell within the network, the UE still attempts to connect with the cell according to the procedures defined in the 3GPP specification. In other words, even if the UE has previously encountered incorrect or non-optimized configurations for the cell, the UE is required to try and connect to the cell even though the UE will continue to encounter the issues caused by the 2 Kuo: Applying Deep Learning in User Equipment Measurable KPIs to Avoid Published by Technical Disclosure Commons, 2021 incorrect or non-optimized configurations for the cell. For example, an NR cell can be configured and allocated to a UE in a non-standalone (NSA) network. If the UE is too far from the NR cell, the UE cannot establish a link with the allocated NR cell and will experience an NR RACH failure.',
  'text_for_rank': 'As s

## Extractive fallback answerer

This fallback returns **one answer object**. It is deliberately simple: it concatenates the best diverse evidence snippets and cites the documents those snippets came from.

There is no citation target. Citations come from the evidence actually used. If fewer than two unique documents are cited, the function backfills with the next-best ranked evidence document only to satisfy the task format requirement.


In [ ]:
def dedupe_keep_order(items: List[int]) -> List[int]:
    out = []
    seen = set()
    for x in items:
        x = int(x)
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

def trim_text(text: str, max_chars: int = MAX_ANSWER_CHARS) -> str:
    text = normalize_text(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0].rstrip(" ,;:") + "..."

def top_unique_docs_from_passages(ranked_passages: List[dict]) -> List[int]:
    """Return candidate document indices ordered by best passage score."""
    if not ranked_passages:
        return []

    tmp = pd.DataFrame(ranked_passages).copy()
    if "doc_score" not in tmp.columns:
        tmp["doc_score"] = tmp["score"] if "score" in tmp.columns else 0.0

    doc_best = (
        tmp.sort_values(["doc_score", "score"], ascending=False)
           .groupby("ref_idx", as_index=False)
           .first()
           .sort_values(["doc_score", "score"], ascending=False)
    )
    return [int(x) for x in doc_best["ref_idx"].tolist()]

def finalize_citations(citations: List[int]) -> List[int]:
    """Deduplicate citations while preserving order.

    There is no hard minimum or maximum citation count. Citation count is determined
    only by support thresholds in citations_for_generated_answer().
    """
    return dedupe_keep_order(citations)

def citations_for_generated_answer(answer_text: str, ranked_passages: List[dict]) -> List[int]:
    """Assign citations only when evidence support is meaningful.

    No minimum or maximum citation count is enforced.
    """
    if not ranked_passages:
        return []

    try:
        claims = [normalize_text(s) for s in sent_tokenize(normalize_text(answer_text))]
    except Exception:
        claims = re.split(r"(?<=[.!?])\s+", normalize_text(answer_text))

    claims = [c for c in claims if len(c.split()) >= 6]
    if not claims:
        return []

    cand = pd.DataFrame(ranked_passages).copy()
    refs = []

    for claim in claims[:MAX_SENTENCES_IN_ANSWER]:
        lex = cand["text_for_rank"].apply(
            lambda x: lexical_overlap(claim, x) if isinstance(x, str) else 0.0
        ).to_numpy(dtype=float)

        support = norm01(lex)

        if USE_EMBED_RERANK:
            try:
                embedder = get_embedder()
                c_emb = embedder.encode([claim], normalize_embeddings=True, show_progress_bar=False)
                p_emb = embedder.encode(
                    cand["text_for_rank"].tolist(),
                    normalize_embeddings=True,
                    batch_size=32,
                    show_progress_bar=False,
                )
                sim = p_emb @ c_emb[0]
                support = 0.35 * norm01(lex) + 0.65 * norm01(sim)
            except Exception:
                pass

        cand["claim_support"] = support
        best = cand.sort_values(["claim_support", "score"], ascending=False).head(CLAIM_TOP_K)


        if len(best) == 0:
            continue

        first = best.iloc[0]
        first_score = float(first["claim_support"])

        if first_score >= CLAIM_SUPPORT_THRESHOLD:
            refs.append(int(first["ref_idx"]))

        # Add additional citations when they are from different docs and still
        # meaningfully support the claim. This is not a minimum citation rule.
        # It just makes multi-source support more likely to be reflected.
        for _, row in best.iloc[1:].iterrows():
            row_ref = int(row["ref_idx"])
            row_score = float(row["claim_support"])

            if (
                row_ref != int(first["ref_idx"])
                and row_ref not in refs
                and row_score >= CLAIM_SECOND_SUPPORT_THRESHOLD
                and row_score >= CLAIM_SECOND_RELATIVE_THRESHOLD * first_score
            ):
                refs.append(row_ref)

    return finalize_citations(refs)

def build_extractive_answer(query: str, ranked_passages: List[dict]) -> List[dict]:
    if not ranked_passages:
        return [{"text": "Insufficient evidence in the provided candidate documents.", "citations": []}]

    snippets = []
    citations = []

    # Use up to a few diverse snippets. No arbitrary citation target; cite what is used.
    for p in ranked_passages:
        ridx = int(p["ref_idx"])
        if ridx in citations and len(snippets) >= 2:
            continue

        snippets.append(trim_text(p["text"], 240))
        citations.append(ridx)

        if len(snippets) >= min(MAX_SENTENCES_IN_ANSWER, 3):
            break

    text = " ".join(snippets)
    citations = finalize_citations(citations)

    return [{"text": trim_text(text), "citations": citations}]


## Evidence-grounded LLM answerer

This baseline generates a short answer from the ranked BM25/embedding evidence.

Important constraints:

- one answer paragraph per query
- no citation target
- citations are assigned from the chunks that best support the generated answer
- the only forced citation behavior is satisfying the minimum two-citation requirement when possible


In [ ]:
GENERATOR = None
TOKENIZER = None
MODEL = None

def load_generator():
    global GENERATOR, TOKENIZER, MODEL
    if MODEL is not None and TOKENIZER is not None:
        return TOKENIZER, MODEL

    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    TOKENIZER = AutoTokenizer.from_pretrained(
        GEN_MODEL_NAME,
        use_fast=True,
    )

    if TOKENIZER.pad_token is None:
        TOKENIZER.pad_token = TOKENIZER.eos_token

    MODEL = AutoModelForCausalLM.from_pretrained(
        GEN_MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )

    MODEL.eval()
    return TOKENIZER, MODEL

def make_generation_messages(query: str, ranked_passages: List[dict]) -> List[dict]:
    evidence_lines = []

    for i, p in enumerate(ranked_passages[:TOP_PASSAGES_FINAL], start=1):
        title = normalize_text(p.get("title", ""))
        evidence = trim_text(p["text"], MAX_CHARS_PER_EVIDENCE)

        # Use simple labels that are less likely to leak into the answer.
        evidence_lines.append(
            f"Passage {i}:\n"
            f"Document title: {title}\n"
            f"Relevant text: {evidence}"
        )

    evidence_block = "\n\n".join(evidence_lines)

    system = (
        "You are a scientific RAG answer writer. Answer only from the supplied evidence. "
        "Write in English only. "
        "Do not use outside knowledge. Do not invent methods, datasets, years, or results. "
        "Do not copy long passages. Do not include paper titles. "
        "Do not mention evidence IDs."
        "Write a concise answer that directly answers the question."
        "Return only the final answer in English."
    )

    user = f"""Question:
    {query}

    Evidence:
    {evidence_block}

    Write an answer in 1-3 sentences.

    Rules:
    - No bullets.
    - No markdown.
    - No paper titles.
    - No citations in the text.
    - Do not repeat the same idea.
    - Do not start with the title of a paper.
    - If multiple evidence passages say the same thing, synthesize them once.
    - Write in English only.
    - Use standard English spellings for chemical names and technical terms.
    """

    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def clean_generated_answer(text: str) -> str:
    text = normalize_text(text)
    # Remove common accidental wrappers.
    text = re.sub(r"^(Answer:|Final answer:)\s*", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```.*?```", "", text, flags=re.DOTALL).strip()
    text = text.replace("[/INST]", "").strip()

    # Keep only first few sentences to avoid drift.
    try:
        sents = [normalize_text(s) for s in sent_tokenize(text)]
    except Exception:
        sents = re.split(r"(?<=[.!?])\s+", text)
    sents = [s for s in sents if s and not s.lower().startswith(("question:", "evidence:", "instructions:"))]
    text = " ".join(sents[:MAX_SENTENCES_IN_ANSWER])
    return trim_text(text, MAX_ANSWER_CHARS)

def generate_raw_answer(query: str, ranked_passages: List[dict]) -> str:
    import torch

    tokenizer, model = load_generator()
    messages = make_generation_messages(query, ranked_passages)

    if hasattr(tokenizer, "apply_chat_template"):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        prompt = messages[0]["content"] + "\n\n" + messages[1]["content"] + "\n\nAnswer:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.08,
            no_repeat_ngram_size=4,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    out = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return clean_generated_answer(out)

def answer_looks_bad(answer_text: str, query: str) -> bool:
    a = normalize_unicode_answer(answer_text)
    al = a.lower()
    q = normalize_text(query).lower()

    if len(a.split()) < 10:
        return True

    if len(a.split()) > 85:
        return True

    if al.startswith(q[:80]):
        return True

    if contains_foreign_script(a):
        return True

    bad_markers = [
        "question:",
        "evidence:",
        "evidence passages:",
        "citation_index",
        "doc_score",
        "crag_score",
        "[e",
        "passage 1:",
        "passage 2:",
        "document title:",
        "relevant text:",
    ]

    if any(marker in al for marker in bad_markers):
        return True

    if "://" in al or "www." in al:
        return True

    if a.count("...") >= 1:
        return True

    return False

def generate_llm_answer(query: str, ranked_passages: List[dict]) -> List[dict]:
    if not ranked_passages:
        return [{"text": "Insufficient evidence in the provided candidate documents.", "citations": []}]

    answer_text = generate_raw_answer(query, ranked_passages)

    # Retry once if Qwen produces foreign-script text, prompt leakage, URLs, or metadata copying.
    if answer_looks_bad(answer_text, query):
        retry_query = (
            query
            + "\n\nImportant: answer in English only. Do not use Cyrillic, Chinese, or non-English text. "
              "Do not copy metadata, URLs, passage labels, or bibliography text. "
              "Write a clean 1-2 sentence answer."
        )
        answer_text = generate_raw_answer(retry_query, ranked_passages)

    if answer_looks_bad(answer_text, query):
        return build_extractive_answer(query, ranked_passages)

    citations = citations_for_generated_answer(answer_text, ranked_passages)
    return [{"text": answer_text, "citations": citations}]

## Build one TIRA / TREC-RAG run line

Every output line has:

- `metadata`
- `references`
- `answer`

The baseline keeps `answer` as a list of exactly one object. The citation indices point into `references`.


In [ ]:
def serialize_doc_id(x: Any):
    """Keep numeric doc ids numeric when possible, matching the organizer example.
    Accessors still use str(doc_id) internally."""
    s = str(x)
    if re.fullmatch(r"\d+", s):
        return int(s)
    return s

def validate_run_entry(entry: dict):
    assert "metadata" in entry and "references" in entry and "answer" in entry
    md = entry["metadata"]
    for k in ["team_id", "run_id", "type", "narrative_id", "narrative"]:
        assert k in md, f"missing metadata.{k}"

    assert isinstance(entry["references"], list)
    assert isinstance(entry["answer"], list)

    # Current advisor/task interpretation: one answer object per query.
    assert len(entry["answer"]) == 1, "Task 4 run should contain exactly one answer object per query."

    seg = entry["answer"][0]
    assert "text" in seg and "citations" in seg
    assert isinstance(seg["text"], str) and len(seg["text"].strip()) > 0
    assert isinstance(seg["citations"], list)

    for c in seg["citations"]:
        assert isinstance(c, int)
        assert 0 <= c < len(entry["references"]), f"citation {c} out of range"

def build_run_entry(qrow: pd.Series, accessor) -> dict:
    # Use string IDs for lookup/ranking; serialize in references for output.
    doc_ids_for_lookup = [str(x) for x in qrow["doc_ids"]]
    references = [serialize_doc_id(x) for x in qrow["doc_ids"]]

    ranked = rank_candidate_passages(qrow["query"], doc_ids_for_lookup, accessor, top_k=TOP_PASSAGES_FINAL)

    if USE_LLM:
        try:
            answer = generate_llm_answer(qrow["query"], ranked)
        except Exception as e:
            print(f"LLM generation failed for {qrow['query_id']}: {e}")
            gc.collect()
            answer = build_extractive_answer(qrow["query"], ranked)
    else:
        answer = build_extractive_answer(qrow["query"], ranked)

    # Final defensive citation repair/format cleanup.
    answer_text = trim_text(answer[0]["text"], MAX_ANSWER_CHARS)
    citations = finalize_citations(answer[0].get("citations", []))

    entry = {
        "metadata": {
            "team_id": TEAM_ID,
            "run_id": RUN_ID,
            "type": RUN_TYPE,
            "narrative_id": str(qrow["query_id"]),
            "narrative": qrow["query"],
        },
        "references": references,
        "answer": [{"text": answer_text, "citations": citations}],
    }

    validate_run_entry(entry)
    return entry

sample_entry = build_run_entry(queries_df.iloc[0], docs_accessor)
sample_entry

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{'metadata': {'team_id': 'LongEval DS@GT',
  'run_id': 'baseline-bm25-bge-Qwen2-7b',
  'type': 'automatic',
  'narrative_id': 'aa42e210a361571ff4d1fad892b75d15',
  'narrative': 'How can a device avoid futile access attempts while still selecting connectivity for best throughput?'},
 'references': [275699672,
  275699915,
  122371639,
  173704007,
  157915138,
  163088229,
  148414038,
  268301,
  303362068,
  129069349],
 'answer': [{'text': 'The device can avoid futile access while selecting connectivity for optimal throughput by using a deep learning algorithm to predict better DL throughput conditions. The UE records past throughput experiences and trains a model to decide between LTE-only and 5G dual connectivity modes, thereby optimizing connection choices.',
   'citations': [9]}]}

In [ ]:
# Generate the complete run
run_entries = []
for _, qrow in tqdm(queries_df.iterrows(), total=len(queries_df)):
    run_entries.append(build_run_entry(qrow, docs_accessor))

with open(RUN_JSONL, "w") as f:
    for entry in run_entries:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print("Saved run to:", RUN_JSONL)
print("Queries written:", len(run_entries))

  0%|          | 0/47 [00:00<?, ?it/s]

Saved run to: /content/drive/MyDrive/GATech/GT Data Science/ARC-CLEF/Outputs/baseline-bm25-bge-Qwen2-7b.jsonl
Queries written: 47


## Error analysis helpers

Use these checks before submitting:

- exactly one answer object per query
- citation indices are in range
- at least two unique citations when possible
- no empty answers
- no prompt echoing


In [ ]:
def inspect_examples(run_jsonl: Path, n: int = 5):
    rows = []
    with open(run_jsonl, "r") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            rows.append(json.loads(line))
    return rows

def audit_run(run_jsonl: Path) -> pd.DataFrame:
    problems = []
    cite_counter = {}
    n = 0

    with open(run_jsonl, "r") as f:
        for line in f:
            n += 1
            obj = json.loads(line)
            qid = obj["metadata"]["narrative_id"]
            ans = obj["answer"]
            refs = obj["references"]

            if len(ans) != 1:
                problems.append((qid, "answer_count", len(ans)))

            text = ans[0].get("text", "")
            cites = ans[0].get("citations", [])

            bad = [c for c in cites if not isinstance(c, int) or c < 0 or c >= len(refs)]
            if bad:
                problems.append((qid, "citation_out_of_range", bad))

            if answer_looks_bad(text, obj["metadata"]["narrative"]):
                problems.append((qid, "bad_or_echoed_answer", text[:120]))

            for c in cites:
                cite_counter[c] = cite_counter.get(c, 0) + 1

    print(f"Audited {n} entries.")
    print("Citation index usage:", dict(sorted(cite_counter.items())))
    return pd.DataFrame(problems, columns=["query_id", "problem", "value"])

examples = inspect_examples(RUN_JSONL, 2)
audit = audit_run(RUN_JSONL)
examples, audit.head(20)

Audited 47 entries.
Citation index usage: {0: 2, 1: 1, 2: 7, 3: 8, 4: 8, 5: 8, 6: 4, 7: 3, 8: 9, 9: 15}


([{'metadata': {'team_id': 'LongEval DS@GT',
    'run_id': 'baseline-bm25-bge-Qwen2-7b',
    'type': 'automatic',
    'narrative_id': 'aa42e210a361571ff4d1fad892b75d15',
    'narrative': 'How can a device avoid futile access attempts while still selecting connectivity for best throughput?'},
   'references': [275699672,
    275699915,
    122371639,
    173704007,
    157915138,
    163088229,
    148414038,
    268301,
    303362068,
    129069349],
   'answer': [{'text': 'The device can avoid futile access while selecting connectivity for optimal throughput by using a deep learning algorithm to predict better DL throughput conditions. The UE records past throughput experiences and trains a model to decide between LTE-only and 5G dual connectivity modes, thereby optimizing connection choices.',
     'citations': [9]}]},
  {'metadata': {'team_id': 'LongEval DS@GT',
    'run_id': 'baseline-bm25-bge-Qwen2-7b',
    'type': 'automatic',
    'narrative_id': '46395a3cf66a9f6a75c89354410d1493